# Notebook 03: Multi-Criteria Prioritization Scoring & Ranking

## 1. Objective
This notebook implements **Decisions 4 & 5** of the assignment:
1. Applying our explainable Multi-Criteria Decision Analysis (MCDA) model across the 30 candidate SKUs.
2. Calculating subscores for all 5 dimensions under the approved Primary Baseline weights:
   - **F1: Demand Evidence (25%)**
   - **F2: International Procurement Delay (25%)**
   - **F3: Vehicle Compatibility (20%)**
   - **F4: Sourcing Practicality (20%)**
   - **F5: Evidence Confidence (10%)**
3. Generating the composite priority score and identifying the **Top 5 Priority Parts** for pilot importation.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Add project root to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.config import PROCESSED_DATA_DIR, RANKINGS_DIR, BASELINE_SCORING_WEIGHTS
print("Approved Baseline Weights:", BASELINE_SCORING_WEIGHTS)

## 2. Ingesting Stage 5C Reconciled Master Dataset

In [ ]:
dataset_path = PROCESSED_DATA_DIR / 'stage5c_reconciled_master_dataset.csv'
if not dataset_path.exists():
    dataset_path = PROCESSED_DATA_DIR / 'ev_parts_master_dataset.csv'
df = pd.read_csv(dataset_path)
print(f"Candidate dataset loaded: {len(df)} SKUs across {df['vehicle_model'].nunique()} vehicles")
df[['sku_id', 'vehicle_model', 'part_category', 'f1_demand_wear', 'f2_procurement_delay', 'f3_compatibility', 'f4_sourcing_practicality', 'f5_evidence_confidence']].head(10)

## 3. Scoring Engine Execution & Ranking

In [ ]:
# Normalize F5 to 100 scale (stored as 1-5 scale in table)
df['f5_norm'] = df['f5_evidence_confidence'] * 20.0

df['composite_score'] = (
    0.25 * df['f1_demand_wear'] +
    0.25 * df['f2_procurement_delay'] +
    0.20 * df['f3_compatibility'] +
    0.20 * df['f4_sourcing_practicality'] +
    0.10 * df['f5_norm']
)

df_ranked = df.sort_values(by=['composite_score', 'f1_demand_wear', 'f4_sourcing_practicality', 'sku_id'], ascending=[False, False, False, True]).reset_index(drop=True)
df_ranked['rank'] = range(1, len(df_ranked) + 1)
print("Scoring execution complete. Top 5 candidates:")
df_ranked[['rank', 'sku_id', 'vehicle_model', 'part_category', 'composite_score']].head(5)

## 4. Top 5 Provisional Recommendations Brief

In [ ]:
top_5 = df_ranked.head(5)
for idx, row in top_5.iterrows():
    print(f"Rank {row['rank']}: {row['sku_id']} | {row['vehicle_model']} {row['part_category']} | Score: {row['composite_score']:.2f}")